<a href="https://colab.research.google.com/github/asennakesavan/InceptezGenAI-Batch26/blob/main/Smart_building_intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer #Impute the missing values with MEAN / MEDIAN / MODE
from sklearn.preprocessing import  StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score,precision_score, recall_score, f1_score, confusion_matrix,classification_report)
from imblearn.over_sampling import SMOTE

In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [8]:
train_df.head()

,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR,Room_Occupancy_Count
0,2017/12/23,14:55:37,25.81,26.88,25.50,26.25,12,13,62,45,0.09,0.08,0.09,0.08,535,0.438462,0,0,0
1,2017/12/25,09:31:36,25.06,25.06,24.56,25.38,7,8,38,25,0.07,0.04,0.06,0.11,355,0.000000,0,0,0
2,2017/12/26,07:27:23,25.06,25.06,24.56,25.25,1,1,10,6,0.08,0.05,0.06,0.09,350,0.000000,0,0,0
3,2017/12/22,17:59:35,26.31,26.50,25.94,26.38,148,235,178,10,1.73,0.09,0.17,0.07,930,2.588462,1,1,3
4,2018/01/10,19:23:13,25.63,25.63,25.31,25.69,0,0,0,0,0.07,0.05,0.05,0.09,545,-2.130769,0,0,0


In [9]:
test_df.head()

,id,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR
0,ROOM-00001,2017/12/23,14:08:08,26.13,27.13,25.50,26.38,160,247,75,56,0.56,0.90,0.30,0.24,640,1.415385,0,0
1,ROOM-00002,2017/12/23,07:25:17,25.06,25.06,24.44,25.38,1,1,10,6,0.08,0.06,0.06,0.06,355,0.000000,0,0
2,ROOM-00003,2018/01/11,02:46:25,25.19,25.19,24.69,25.25,0,0,0,0,0.07,0.05,0.05,0.08,345,0.000000,0,0
3,ROOM-00004,2017/12/24,03:01:40,25.31,25.31,24.75,25.69,0,0,0,0,0.07,0.05,0.06,0.07,360,-0.096154,0,0
4,ROOM-00005,2017/12/23,19:57:53,26.19,26.00,25.88,26.31,0,0,0,0,0.07,0.05,0.05,0.07,1010,0.896154,0,0


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8103 entries, 0 to 8102
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  8103 non-null   object 
 1   Time                  8103 non-null   object 
 2   S1_Temp               8103 non-null   float64
 3   S2_Temp               8103 non-null   float64
 4   S3_Temp               8103 non-null   float64
 5   S4_Temp               8103 non-null   float64
 6   S1_Light              8103 non-null   int64  
 7   S2_Light              8103 non-null   int64  
 8   S3_Light              8103 non-null   int64  
 9   S4_Light              8103 non-null   int64  
 10  S1_Sound              8103 non-null   float64
 11  S2_Sound              8103 non-null   float64
 12  S3_Sound              8103 non-null   float64
 13  S4_Sound              8103 non-null   float64
 14  S5_CO2                8103 non-null   int64  
 15  S5_CO2_Slope         

In [5]:
print(train_df.shape)
print(test_df.shape)

(8103, 19)
(2026, 19)


In [6]:
train_df.isna().sum()

,0
Date,0
Time,0
S1_Temp,0
S2_Temp,0
S3_Temp,0
S4_Temp,0
S1_Light,0
S2_Light,0
S3_Light,0
S4_Light,0


In [7]:
test_df.isna().sum()

,0
id,0
Date,0
Time,0
S1_Temp,0
S2_Temp,0
S3_Temp,0
S4_Temp,0
S1_Light,0
S2_Light,0
S3_Light,0


In [11]:
duplicates = train_df[train_df.duplicated()]
print(duplicates)

Empty DataFrame
Columns: [Date, Time, S1_Temp, S2_Temp, S3_Temp, S4_Temp, S1_Light, S2_Light, S3_Light, S4_Light, S1_Sound, S2_Sound, S3_Sound, S4_Sound, S5_CO2, S5_CO2_Slope, S6_PIR, S7_PIR, Room_Occupancy_Count]
Index: []


In [31]:
train_df[['S1_Temp','S2_Temp','S3_Temp','S4_Temp','S1_Light','S2_Light','S3_Light','S4_Light','S1_Sound','S2_Sound','S3_Sound','S4_Sound','S5_CO2','S5_CO2_Slope','S6_PIR','S7_PIR']].agg(['min', 'max'])

,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR
min,24.94,24.75,24.44,25.0,0,0,0,0,0.06,0.04,0.04,0.05,345,-6.296154,0,0
max,26.38,29.00,26.19,26.5,165,258,280,74,3.88,3.44,3.67,3.40,1270,8.980769,1,1


In [33]:
train_df['Room_Occupancy_Count'].value_counts().sort_index()

,count
Room_Occupancy_Count,
0,6582
1,367
2,599
3,555


In [38]:
train_df.groupby('Room_Occupancy_Count')[['S1_Temp','S2_Temp','S3_Temp','S4_Temp','S5_CO2','S5_CO2_Slope','S1_Sound','S2_Sound','S3_Sound','S4_Sound']].agg(['max','min'])

S1_Temp        S2_Temp        S3_Temp        S4_Temp  \
                         max    min     max    min     max    min     max   
Room_Occupancy_Count                                                        
0                      26.31  25.00   27.25  25.00   26.13  24.44   26.44   
1                      26.13  24.94   26.81  24.75   25.69  24.50   26.44   
2                      26.31  25.38   28.69  25.31   26.00  24.81   26.44   
3                      26.38  25.50   29.00  25.63   26.19  25.19   26.50   

                            S5_CO2      S5_CO2_Slope           S1_Sound        \
                        min    max  min          max       min      max   min   
Room_Occupancy_Count                                                            
0                     25.00   1250  345     2.961538 -6.296154     3.88  0.06   
1                     25.38    630  360     2.800000 -3.634615     3.84  0.06   
2                     25.69   1065  380     5.138462 -2.761538     3.80  0.06   
3                     25.63   1270  370     8.980769 -0.911538     3.75  0.06   

                     S2_Sound       S3_Sound       S4_Sound        
                          max   min      max   min      max   min  
Room_Occupancy_Count                                               
0                        3.40  0.04     3.67  0.04     1.80  0.05  
1                        1.12  0.04     0.68  0.05     0.39  0.05  
2                        3.44  0.04     3.66  0.05     1.70  0.06  
3                        3.44  0.04     3.66  0.06     3.40  0.06

In [41]:
Numerical_features = train_df.select_dtypes(include=['float64','int64']).drop(columns='Room_Occupancy_Count')

In [42]:
Numerical_features

,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR
0,25.81,26.88,25.50,26.25,12,13,62,45,0.09,0.08,0.09,0.08,535,0.438462,0,0
1,25.06,25.06,24.56,25.38,7,8,38,25,0.07,0.04,0.06,0.11,355,0.000000,0,0
2,25.06,25.06,24.56,25.25,1,1,10,6,0.08,0.05,0.06,0.09,350,0.000000,0,0
3,26.31,26.50,25.94,26.38,148,235,178,10,1.73,0.09,0.17,0.07,930,2.588462,1,1
4,25.63,25.63,25.31,25.69,0,0,0,0,0.07,0.05,0.05,0.09,545,-2.130769,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8098,25.44,25.44,25.19,26.06,0,0,0,0,0.07,0.05,0.07,0.11,355,0.230769,0,0
8099,25.06,25.06,24.56,25.44,7,8,37,24,0.08,0.05,0.06,0.11,355,0.000000,0,0
8100,25.06,25.06,24.56,25.13,0,0,5,3,0.08,0.05,0.06,0.09,345,0.000000,0,0
8101,25.19,25.19,24.69,25.50,0,0,0,0,0.08,0.05,0.06,0.06,360,0.000000,0,0


In [48]:
x = train_df[Numerical_features.columns]
y= train_df['Room_Occupancy_Count']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=19,stratify=y)
print(f"train: {x_train.shape}, test: {x_test.shape}")

train: (6482, 16), test: (1621, 16)


In [98]:
scaler = StandardScaler()
x_train_scaled = pd.DataFrame(scaler.fit_transform(x_train),columns = Numerical_features.columns,index = x_train.index)
x_test_scaled = pd.DataFrame(scaler.fit_transform(x_test),columns = Numerical_features.columns,index = x_test.index)

In [74]:
results = []
log_reg = LogisticRegression(random_state=19, max_iter=1000, class_weight='balanced')
tree = DecisionTreeClassifier(random_state=19, class_weight='balanced')
knn = KNeighborsClassifier(n_neighbors=5)

log_reg.fit(x_train_scaled, y_train)
tree.fit(x_train_scaled, y_train)
knn.fit(x_train_scaled,y_train)

for name, model in [('class_weight · Logistic Regression', log_reg), ('class_weight · Decision Tree', tree), ('class_weight · Knn', knn)]:
    y_pred = model.predict(x_test_scaled)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (macro)': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall (macro)': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1 (macro)': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'F1 (weighted)': f1_score(y_test, y_pred, average='weighted', zero_division=0),

    })

In [61]:
smallest_class_size = y_train.value_counts().min()
safe_k = max(1, min(5, smallest_class_size - 1))

smote_multi = SMOTE(random_state=42, k_neighbors=safe_k)
X_train_m_sm, y_train_multi_sm = smote_multi.fit_resample(x_train_scaled, y_train)
print(y_train_multi_sm.value_counts())

Room_Occupancy_Count
0    5265
2    5265
3    5265
1    5265
Name: count, dtype: int64


In [75]:
log_reg_sm = LogisticRegression(random_state=19, max_iter=1000)
knn_sm = KNeighborsClassifier(n_neighbors=5)
tree_sm = DecisionTreeClassifier(random_state=19)

log_reg_sm.fit(X_train_m_sm, y_train_multi_sm)
knn_sm.fit(X_train_m_sm, y_train_multi_sm)
tree_sm.fit(X_train_m_sm, y_train_multi_sm)

for name, model in [('SMOTE · Logistic Regression', log_reg_sm), ('SMOTE · KNN', knn_sm), ('SMOTE · Decision Tree', tree_sm)]:
    y_pred = model.predict(x_test_scaled)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1 (macro)': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'F1 (weighted)': f1_score(y_test, y_pred, average='weighted', zero_division=0),
        'Recall (macro)': recall_score(y_test, y_pred, average='macro', zero_division=0),
    })



In [76]:
results_df = pd.DataFrame(results).set_index('Model')
results_df

,Accuracy,Precision (macro),Recall (macro),F1 (macro),F1 (weighted)
Model,,,,,
class_weight · Logistic Regression,0.993214,0.975014,0.982088,0.978520,0.993243
class_weight · Decision Tree,0.994448,0.977031,0.981802,0.979095,0.994435
class_weight · Knn,0.991980,0.973079,0.972839,0.972802,0.991984
SMOTE · Logistic Regression,0.993831,NaN,0.982278,0.979686,0.993845
SMOTE · KNN,0.990130,NaN,0.966082,0.966242,0.990122
SMOTE · Decision Tree,0.988896,NaN,0.962208,0.958935,0.988877


In [78]:
test_df.head()

,id,Date,Time,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR
0,ROOM-00001,2017/12/23,14:08:08,26.13,27.13,25.50,26.38,160,247,75,56,0.56,0.90,0.30,0.24,640,1.415385,0,0
1,ROOM-00002,2017/12/23,07:25:17,25.06,25.06,24.44,25.38,1,1,10,6,0.08,0.06,0.06,0.06,355,0.000000,0,0
2,ROOM-00003,2018/01/11,02:46:25,25.19,25.19,24.69,25.25,0,0,0,0,0.07,0.05,0.05,0.08,345,0.000000,0,0
3,ROOM-00004,2017/12/24,03:01:40,25.31,25.31,24.75,25.69,0,0,0,0,0.07,0.05,0.06,0.07,360,-0.096154,0,0
4,ROOM-00005,2017/12/23,19:57:53,26.19,26.00,25.88,26.31,0,0,0,0,0.07,0.05,0.05,0.07,1010,0.896154,0,0


In [79]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2026 entries, 0 to 2025
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            2026 non-null   object 
 1   Date          2026 non-null   object 
 2   Time          2026 non-null   object 
 3   S1_Temp       2026 non-null   float64
 4   S2_Temp       2026 non-null   float64
 5   S3_Temp       2026 non-null   float64
 6   S4_Temp       2026 non-null   float64
 7   S1_Light      2026 non-null   int64  
 8   S2_Light      2026 non-null   int64  
 9   S3_Light      2026 non-null   int64  
 10  S4_Light      2026 non-null   int64  
 11  S1_Sound      2026 non-null   float64
 12  S2_Sound      2026 non-null   float64
 13  S3_Sound      2026 non-null   float64
 14  S4_Sound      2026 non-null   float64
 15  S5_CO2        2026 non-null   int64  
 16  S5_CO2_Slope  2026 non-null   float64
 17  S6_PIR        2026 non-null   int64  
 18  S7_PIR        2026 non-null 

In [92]:
test_ds_Numerical_features = test_df.select_dtypes(include=['float64','int64']).columns

In [93]:
test_ds_Numerical_features

Index(['S1_Temp', 'S2_Temp', 'S3_Temp', 'S4_Temp', 'S1_Light', 'S2_Light',
       'S3_Light', 'S4_Light', 'S1_Sound', 'S2_Sound', 'S3_Sound', 'S4_Sound',
       'S5_CO2', 'S5_CO2_Slope', 'S6_PIR', 'S7_PIR'],
      dtype='object')

In [90]:
test_df['S1_Temp'].value_counts()

,count
S1_Temp,
25.44,233
25.19,223
25.06,205
25.31,194
25.13,192
25.38,190
25.25,143
26.19,65
25.50,64


In [88]:
test_df.describe()

,S1_Temp,S2_Temp,S3_Temp,S4_Temp,S1_Light,S2_Light,S3_Light,S4_Light,S1_Sound,S2_Sound,S3_Sound,S4_Sound,S5_CO2,S5_CO2_Slope,S6_PIR,S7_PIR
count,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000,2026.000000
mean,25.446851,25.534348,25.047892,25.744822,25.012833,25.568608,32.940276,13.347976,0.169768,0.122591,0.149072,0.101061,455.488648,0.032387,0.095755,0.080948
std,0.344529,0.582696,0.419155,0.353960,50.627576,66.616133,56.087629,19.737083,0.308442,0.279609,0.369853,0.098596,194.854699,1.212522,0.294328,0.272822
min,24.940000,24.750000,24.440000,24.940000,0.000000,0.000000,0.000000,0.000000,0.060000,0.040000,0.050000,0.050000,345.000000,-6.296154,0.000000,0.000000
25%,25.190000,25.190000,24.690000,25.440000,0.000000,0.000000,0.000000,0.000000,0.070000,0.050000,0.060000,0.060000,355.000000,-0.046154,0.000000,0.000000
50%,25.380000,25.380000,24.940000,25.750000,0.000000,0.000000,0.000000,0.000000,0.080000,0.050000,0.060000,0.090000,360.000000,0.000000,0.000000,0.000000
75%,25.630000,25.630000,25.310000,26.000000,11.750000,14.000000,49.000000,22.000000,0.080000,0.060000,0.070000,0.100000,445.000000,0.000000,0.000000,0.000000
max,26.380000,28.940000,26.190000,26.560000,165.000000,257.000000,279.000000,74.000000,3.830000,3.440000,3.670000,1.840000,1270.000000,8.946154,1.000000,1.000000


In [95]:
x_test_ds_scaled = pd.DataFrame(scaler.fit_transform(test_df[test_ds_Numerical_features]),columns = test_ds_Numerical_features,index = test_df.index)

In [104]:
y_pred_test = tree.predict(x_test_ds_scaled)

In [106]:
Final_prediction = pd.DataFrame({
    'id' : test_df['id'],
    'prediction' : y_pred_test
}

)

In [108]:
Final_prediction.to_csv("Final_prediction.csv")